# Multi-Agent Observability & Evaluation — Demo

### A production-style showcase of how to *see* and *measure* AI agents

---

## What is this?

An end-to-end framework that runs AI **web-navigation agents** against the
[Mind2Web](https://osu-nlp-group.github.io/Mind2Web/) benchmark (NeurIPS 2023) and
instruments every step with **OpenTelemetry-compliant tracing** and a rigorous
**multi-layer evaluation** stack. It compares a **single ReAct agent** against a
**Multi-Agent System (MAS)** built on the supervisor pattern:

> **Supervisor → Planner → Navigator → Validator**

## Why does it matter?

As autonomous agents move into production, two questions become mission-critical:

1. **Observability** — *Can you see what the agent did?* Every LLM call, routing
   decision, and tool invocation is captured as an OTel span you can ship to
   Datadog, Splunk, Phoenix, or Langfuse — an auditable record of *how* a result
   was reached, not just *that* it finished.
2. **Evaluation** — *Can you prove it did the right thing?* A hybrid scorer
   (deterministic rules + LLM-as-judge), tool-selection correctness, and safety
   checks quantify quality, cost, latency, and health.

## The story this demo tells

A single agent is simple and cheap, but it has to plan, act, and self-check all at
once. A **Multi-Agent System decomposes the work** — a Planner decides the steps, a
Navigator executes them with tools, and a Validator independently checks the result.
On harder, multi-step tasks this division of labor can **lift task quality** at the
cost of more tokens and latency. This notebook runs **both systems on the same
tasks** and puts the trade-off side by side, so you can decide *when orchestration is
worth it*.

## Value

- **Provider-agnostic** — Azure OpenAI, OpenAI, Ollama, Groq, Together (via `.env`)
- **Audit-ready** — OTLP trace export + per-agent cost attribution
- **Coding-light** — all logic lives in `src/`; this notebook is the narrative
- **Reusable** — swap in your own agents, tools, or benchmark

---


## Step 1 — Install & Import

All implementation lives in `src/` — this notebook stays deliberately coding-light.

| Symbol | Role |
|---|---|
| `Config` | Provider-agnostic LLM factory + cost table |
| `TracingManager`, `HierarchicalTracer` | Per-task trace + OpenTelemetry span tree |
| `CostTracker`, `HealthMonitor` | Cost rollups + rolling-window health |
| `SafetyValidator` | PII / injection / harmful-content checks |
| `load_mind2web`, `Mind2WebTask` | Benchmark loader + task object |
| `create_baseline_agent`, `run_agent` | Single ReAct agent |
| `create_multi_agent`, `run_multi_agent` | Supervisor → Planner → Navigator → Validator |
| `evaluate_batch` | Runs N tasks through either system → tidy DataFrame |
| `HybridEvaluator`, `ToolCorrectnessEval` | Scoring engines |
| `plot_*` | Dashboards, trace tree, single-vs-multi comparison |

> `%autoreload 2` is enabled, so edits under `src/` take effect without a kernel restart.

In [ ]:
# Uncomment on first run:
# !pip install -r requirements.txt

%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '.')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from src import (
    Config,
    TracingManager, HierarchicalTracer,
    CostTracker, HealthMonitor,
    SafetyValidator,
    load_mind2web,
    Mind2WebTask,
    create_baseline_agent, run_agent,
    create_multi_agent, run_multi_agent,
    evaluate_batch,
    HybridEvaluator, ToolCorrectnessEval,
    plot_eval_dashboard, plot_trace_tree, plot_telemetry_dashboard,
    plot_dataset_overview, plot_baseline_vs_multi,
)

print('✅ All imports successful')

## Step 2 — Configuration

Settings come from your `.env` (copied from `.env.example`) — **no hardcoded
credentials**. Provider is auto-detected: `OPENAI_API_VERSION` set → Azure OpenAI;
blank → OpenAI / Ollama / Groq / any compatible endpoint.

**Per-specialist models (optional).** The multi-agent system can give each role its
own model. By default Planner/Navigator/Supervisor use `AGENT_MODEL` and Validator
uses `JUDGE_MODEL`, but you can override in `.env` — e.g. a cheap Planner and a
strong Navigator:

```
PLANNER_MODEL=gpt-4o-mini
NAVIGATOR_MODEL=gpt-4o
VALIDATOR_MODEL=gpt-4.1
```

> **Tip:** use a *different* model for the judge/validator than the agent to reduce
> self-evaluation bias.

In [ ]:
Config.setup_dirs()

print('Provider:    ', 'Azure OpenAI' if Config.API_VERSION else 'OpenAI / compatible')
print('Agent model: ', Config.AGENT_MODEL)
print('Judge model: ', Config.JUDGE_MODEL)
print('MAS models:  ', {
    'supervisor': Config.SUPERVISOR_MODEL, 'planner': Config.PLANNER_MODEL,
    'navigator':  Config.NAVIGATOR_MODEL,  'validator': Config.VALIDATOR_MODEL,
})
print('Pass threshold:', Config.EVAL_PASS_THRESHOLD)
print('Output dir:    ', Config.OUTPUT_DIR)

## Step 3 — Verify LLM Connectivity

A quick smoke test before spending time on the full pipeline — one tiny call to the
agent model and one to the judge model. Catches auth errors, wrong deployment names,
and IP-allowlist blocks *early*. If either errors, fix `.env` (see
[`docs/provider-setup.md`](docs/provider-setup.md)) before continuing.

In [ ]:
from langchain_core.messages import HumanMessage

agent_llm = Config.create_llm(role='agent')
judge_llm = Config.create_llm(role='judge')

print('Agent LLM ✅:', agent_llm.invoke([HumanMessage(content='Say hello in 5 words.')]).content.strip())
print('Judge LLM ✅:', judge_llm.invoke([HumanMessage(content='Reply with the single digit 1.')]).content.strip())

## Step 4 — Load the Mind2Web Benchmark (Test Data)

[**Mind2Web**](https://osu-nlp-group.github.io/Mind2Web/) (NeurIPS 2023, OSU NLP) is
the first large-scale benchmark for agents that follow natural-language instructions
to navigate **real websites** — 2,000+ tasks across 137 sites and 31 domains.

We stream `osunlp/Multimodal-Mind2Web` from HuggingFace, keeping only lightweight
**text metadata** (skipping HTML/screenshots). Each task has a `confirmed_task`, a
`website`/`domain`, and `action_reprs` — the **gold reference action sequence** used
to score tool correctness. First run caches ~300 tasks to `outputs/data/`.

> The framework scores the agent's *plan* against these references in a safe
> sandbox — it does not drive a live browser against the real site.

In [ ]:
raw_tasks = load_mind2web(Config.DATA_DIR, target_tasks=Config.MIND2WEB_TARGET_TASKS)
print(f'Loaded {len(raw_tasks)} tasks. Sample: {raw_tasks[0]["confirmed_task"][:90]}')

In [ ]:
fig = plot_dataset_overview(raw_tasks, save_path=Config.OUTPUT_DIR / 'dataset_overview.png')

## Step 5 — Build Both Systems

We initialize the shared observability/monitoring components, then build **both
architectures** so they can be compared on identical tasks.

**Single-Agent (baseline)** — one ReAct agent with all 11 hybrid tools. Plans, acts,
and answers in a single loop.

**Multi-Agent System (MAS)** — the supervisor pattern, each specialist with its own
role, model, and OTel span:

| Specialist | Role | Tools? |
|---|---|---|
| **Supervisor** | Routes the pipeline | no |
| **Planner** | Decomposes task into 3–5 steps | no |
| **Navigator** | Executes the plan with tools | **yes** |
| **Validator** | Independently judges completion/quality | no |

**Tools** (`src/tools.py`) are *hybrid real + mock*: READ tools fetch live data when
keys are present (Tavily) else realistic mocks; WRITE tools (book/purchase/submit)
are **always mocked** — no real transactions.

In [ ]:
# Shared observability + monitoring
tracing_manager = TracingManager()
hier_tracer     = HierarchicalTracer()
cost_tracker    = CostTracker()
health_monitor  = HealthMonitor(window_size=50)

# Both systems
agent, judge_llm = create_baseline_agent(Config)
multi_agents     = create_multi_agent(Config)

# Shared evaluator
evaluator = HybridEvaluator(judge_llm,
                            pass_threshold=Config.EVAL_PASS_THRESHOLD,
                            rule_weight=Config.RULE_WEIGHT,
                            llm_weight=Config.LLM_WEIGHT)

print('✅ Single-agent + Multi-agent systems ready')
print('   MAS specialists:', list(multi_agents['models'].keys()))

## Step 6 — Dry Run (1 task, both systems)

Verify the full pipeline on a single task for **each** architecture before the batch.
Both emit an OpenTelemetry span tree (`task.execute → … → tool.execute`). Note how the
multi-agent run shows the per-specialist breakdown in its reasoning steps.

In [ ]:
task = Mind2WebTask.from_dict(raw_tasks[0], idx=0)

# Single agent
out_s, tr_s = run_agent(task, agent, tracing_manager, Config, hier_tracer)
print(f'[Single]  latency={tr_s.latency_ms:.0f}ms  tools={len(tr_s.tool_calls)}  cost=${tr_s.total_cost:.5f}')

# Multi-agent
out_m, tr_m = run_multi_agent(task, multi_agents, hier_tracer, tracing_manager, Config)
print(f'[MAS]     latency={tr_m.latency_ms:.0f}ms  tools={len(tr_m.tool_calls)}  cost=${tr_m.total_cost:.5f}')
print('  per-agent:', tr_m.reasoning_steps)

# Reset so the dry run does not pollute the batch metrics
tracing_manager.reset(); health_monitor.reset(); hier_tracer.reset()

## Step 7 — Single-Agent Baseline (N tasks)

Run the baseline ReAct agent over `QUICK_TEST_N` tasks. `evaluate_batch` applies the
full evaluation stack per task — task completion (rule + LLM judge), tool correctness
(precision/recall/F1), safety, plus cost/latency/health — and returns a DataFrame.

This is our **reference point** for the comparison.

In [ ]:
N = Config.QUICK_TEST_N

df_single, _ = evaluate_batch(
    raw_tasks, N, mode='single',
    agent=agent, evaluator=evaluator,
    hier_tracer=hier_tracer, tracing_manager=tracing_manager,
    cost_tracker=cost_tracker, health_monitor=health_monitor, config=Config,
)
df_single.head()

## Step 8 — Multi-Agent System (same N tasks) ⭐

Now the focus: run the **MAS** over the *same* tasks. Each task flows through
Supervisor → Planner → Navigator → Validator, with every specialist traced and
cost-attributed individually.

Watch the per-task line: the MAS typically makes **more tool calls** (the Navigator
follows an explicit plan) and costs **more** (4 LLM roles vs. 1) — the question Step 9
answers is whether that buys **higher task quality**.

In [ ]:
df_multi, _ = evaluate_batch(
    raw_tasks, N, mode='multi',
    multi_agents=multi_agents, evaluator=evaluator,
    hier_tracer=hier_tracer, tracing_manager=tracing_manager,
    config=Config,
)
df_multi.head()

## Step 9 — Single-Agent vs. Multi-Agent: The Comparison

The headline of the demo. We line up both systems on the same tasks across the
dimensions that matter: **task quality, tool correctness, cost, and latency**.

- If the MAS scores **higher** on `task_score` / pass rate, decomposition helped.
- If it costs **more** for little gain, a single agent is the better default.
- The right choice is **task-dependent** — complex, multi-step tasks favor the MAS;
  simple lookups favor the single agent.

In [ ]:
def _avg(df, col):
    return df[col].mean()

rows = []
for label, df in [('Single Agent', df_single), ('Multi-Agent', df_multi)]:
    rows.append({
        'System':       label,
        'Pass rate':    f"{_avg(df,'task_passed')*100:.0f}%",
        'Avg score':    f"{_avg(df,'task_score'):.3f}",
        'Tool F1':      f"{_avg(df,'tool_f1'):.3f}",
        'Avg cost':     f"${_avg(df,'total_cost'):.4f}",
        'Avg latency':  f"{_avg(df,'latency_ms'):.0f}ms",
        'Avg tools':    f"{_avg(df,'n_tool_calls'):.1f}",
    })
comparison = pd.DataFrame(rows).set_index('System')
comparison

In [ ]:
fig = plot_baseline_vs_multi(df_single, df_multi,
                            save_path=Config.OUTPUT_DIR / 'baseline_vs_multi.png')

In [ ]:
# Auto-generated interpretation of the run
d_score = _avg(df_multi,'task_score') - _avg(df_single,'task_score')
d_cost  = _avg(df_multi,'total_cost') - _avg(df_single,'total_cost')
cost_x  = _avg(df_multi,'total_cost') / max(_avg(df_single,'total_cost'), 1e-9)

print('INTERPRETATION')
print('-'*60)
print(f"Task score:  {'MAS higher' if d_score>0 else 'Single higher'} by {abs(d_score):.3f}")
print(f"Cost:        MAS is {cost_x:.1f}x the single-agent cost (+${d_cost:.4f}/task)")
if d_score > 0.02:
    print('\n→ The Multi-Agent System improved task quality. The extra cost')
    print('  buys better planning and an independent validation check —')
    print('  worth it for complex, multi-step tasks.')
elif d_score < -0.02:
    print('\n→ The single agent matched or beat the MAS here. For tasks this')
    print('  simple, orchestration adds cost without a quality payoff.')
else:
    print('\n→ Quality is comparable on this sample. The MAS advantage grows')
    print('  with task complexity — try a harder task slice to see it widen.')

## Step 10 — Observability Deep-Dive (OpenTelemetry)

The other half of the showcase. `hier_tracer` now holds the **multi-agent** spans
from Step 8. Every span carries GenAI Semantic Convention attributes
(`gen_ai.agent.name`, `gen_ai.agent.role`, `gen_ai.request.model`, `gen_ai.usage.*`)
and the export is **OTLP-compliant JSON** — drop it straight into Datadog, Splunk,
Phoenix, Langfuse, or Jaeger.

```
task.execute
├── agent.supervisor.route
├── agent.planner.plan        (role=task_decomposer)
├── agent.navigator.execute   (role=tool_executor)
│   ├── tool.execute
│   └── tool.execute
└── agent.validator.validate  (role=quality_checker)
```

In [ ]:
# Render the multi-agent trace tree for the first task
first_id = list(hier_tracer.traces.keys())[0]
fig = plot_trace_tree(hier_tracer, first_id, save_path=Config.OUTPUT_DIR / 'mas_trace_tree.png')

# Span + cost stats across the run
print('Trace stats:', hier_tracer.get_stats())

In [ ]:
# Export all spans to OTLP JSON (portable to any OTel backend)
path = hier_tracer.save_all_traces(Config.TRACE_DIR)
print(f'OTLP traces exported → {path}')

## Step 11 — Dashboards

Per-system evaluation and telemetry dashboards: score distribution, pass/fail,
tool-F1, cost-vs-latency, token usage, and rolling pass-rate.

In [ ]:
fig = plot_eval_dashboard(df_multi, threshold=Config.EVAL_PASS_THRESHOLD,
                          save_path=Config.OUTPUT_DIR / 'mas_eval_dashboard.png')

In [ ]:
fig = plot_telemetry_dashboard(df_multi, save_path=Config.OUTPUT_DIR / 'mas_telemetry.png')

## Step 12 — Save Results

Persist both result tables and the cost summary. PNG dashboards and OTLP traces are
already in `outputs/` (git-ignored, so results stay local).

In [ ]:
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

df_single.to_csv(Config.OUTPUT_DIR / f'single_agent_{ts}.csv', index=False)
df_multi.to_csv(Config.OUTPUT_DIR / f'multi_agent_{ts}.csv', index=False)
comparison.to_csv(Config.OUTPUT_DIR / f'comparison_{ts}.csv')

print(f'Saved results → {Config.OUTPUT_DIR}')
print('\nCost summary (single-agent run):')
print(cost_tracker.get_summary())